# PLAYGROUND — one section per scenario, everything configurable
Sections: **multi, single** (IID) and **pavia4, sandiego, sandiego2** (scenes).
Each section is ONE cell: edit its `CFG` dict and run — it sweeps every
(rho, seed) pair sequentially, skipping finished keys. Configurable per
section: `front` (std / zca / robust / none), architecture (DART hidden dims
or full DARTS arch), training (lr / wd / grad_clip / batch / epochs /
cosine annealing), n (IID), target_cls + theta (scenes), eval cadence.
Saves everything: fp16 weight snapshots at every eval, best+final states,
final raw scores, full curves in `results_play.json` (key carries scenario,
det, front, n, rho, seed + your `tag`). Zip cell at the end.
NOTE: the frozen paper recipe is std front / wd=0 / clip 1.0 — zca and
robust fronts are exploration-only here.

In [ ]:
!git clone -b camera-ready --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
for f in ('pavia-u.mat', 'Sandiego.mat', 'Sandiego2.mat'):
    assert os.path.exists(f'repro/data/{f}'), f'missing {f}'

In [ ]:
%%writefile run_play.py
"""Playground engine — every scenario, DART/DARTS, configurable front/arch/
training. Reads the JSON named by env CONFIG. Sequential rho x seed sweep,
done-key skip. Outputs: results_play.json + ckpt_play/<key>/."""
import json
import os
import sys
import time

sys.path.insert(0, os.getcwd())

import numpy as np
import torch
import yaml
from tqdm import tqdm

from repro import scenes
from repro.scenes import pavia_protocol as PP
from repro.protocols.spatial import load_cfg
from repro.protocols.iid import load_hsi, build_pools, _pd_at_fa, _auc
from repro.core.data import Whitening, plant_targets, extract_neighborhoods
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.seeding import seed_all
from repro.core.models import ScoreNet
from repro.models.darts.model import DARTS, _NeighborDenoiser

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(True)
CFG = json.load(open(os.environ['CONFIG']))
SCEN = CFG['scenario']
IS_IID = SCEN in ('multi', 'single')
DET = CFG.get('det', 'dart')
assert not (IS_IID and DET == 'darts'), 'DARTS needs a spatial scene'
FRONT = CFG.get('front', 'std')
TAG = CFG.get('tag', '')
TRAIN = CFG['train']
EPOCHS = int(TRAIN['epochs'])
EVAL_EVERY = int(CFG.get('eval_every', 500))
OUT_JSON = 'results_play.json'
CKPT_ROOT = 'ckpt_play'

SP_CFG = load_cfg()
os.makedirs(CKPT_ROOT, exist_ok=True)
_CACHE = {}


def iid_cfg(mode):
    c = yaml.safe_load(open(f'repro/configs/iid_{mode}.yaml'))
    c['dataset'] = 'repro/data/pavia-u.mat'
    return c


def iid_pools(seed, n):
    key = ('iid', seed, n)
    if key not in _CACHE:
        c = iid_cfg(SCEN)
        data, gt = load_hsi(c['dataset'])
        bkg, tgt = build_pools(data, gt.flatten(), c, SCEN)
        s = tgt.mean(axis=0).astype(np.float32)
        rng = np.random.default_rng(seed)
        idx = np.arange(len(bkg)); rng.shuffle(idx)
        shuf = bkg[idx]
        tr = shuf[:n].astype(np.float32)
        te = shuf[-int(c['test_size']):].astype(np.float32)
        planted, labels, _ = plant_targets(
            te, s, float(c['amplitude']), c['target_fraction'],
            model='additive', seed=seed)
        _CACHE[key] = (tr, planted.astype(np.float32),
                       np.asarray(labels), s)
    return _CACHE[key]


def scene_data():
    if 'sc' not in _CACHE:
        sc = scenes.build(SCEN, SP_CFG)
        k = int(SP_CFG['k'])
        for key, flat, shape in (('_te_nbr', sc['te'], sc['te_shape']),
                                 ('_tr_nbr', sc['tr'], sc['tr_shape'])):
            img = torch.tensor(np.asarray(flat, np.float32)
                               .reshape(*shape, -1))
            _, nbr = extract_neighborhoods(img, k)
            sc[key] = nbr.numpy()
        tc = CFG.get('target_cls')
        if SCEN == 'pavia4' and tc is not None:
            sc['sig_use'] = PP.foreign_signature(
                sc['data'], sc['gt'], sc['te'], cls=int(tc)).astype(np.float32)
        else:
            sc['sig_use'] = np.asarray(sc['sig'], np.float32)
        _CACHE['sc'] = sc
    return _CACHE['sc']


def make_front(tr):
    X = np.asarray(tr, np.float64)
    mu = X.mean(0).astype(np.float32)
    if FRONT == 'none':
        return Whitening(mu, np.eye(X.shape[1], dtype=np.float32))
    if FRONT == 'robust':
        med = np.median(X, axis=0)
        mad = np.median(np.abs(X - med), axis=0) * 1.4826
        return Whitening(med.astype(np.float32),
                         np.diag(1.0 / mad).astype(np.float32))
    if FRONT == 'zca':
        lam, V = np.linalg.eigh(np.cov(X, rowvar=False))   # no floor
        return Whitening(mu, (V @ np.diag(1.0 / np.sqrt(lam)) @ V.T)
                         .astype(np.float32))
    return Whitening(mu, np.diag(1.0 / X.std(0)).astype(np.float32))


def run_one(rho, seed):
    n = int(CFG.get('n', 0)) if IS_IID else 'scene'
    tag = f'-{TAG}' if TAG else ''
    key = f'{SCEN}{tag}_{DET}_{FRONT}_n{n}_r{rho}_s{seed}'
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    if key in res:
        print('skip (done):', key); return
    t0 = time.time()
    if IS_IID:
        tr, planted, y, s = iid_pools(seed, int(CFG['n']))
        shape = None
    else:
        sc = scene_data()
        tr, te, s = sc['tr'], sc['te'], sc['sig_use']
        theta = float(CFG.get('theta', 0.15))
        planted, labels, _ = plant_targets(
            te, s, theta, float(SP_CFG['target_fraction']),
            model='additive', seed=seed, spatial_shape=sc['te_shape'],
            edge_guard=int(SP_CFG['edge_guard']))
        planted = planted.astype(np.float32)
        y = np.asarray(labels)
        shape = sc['te_shape']
    D = tr.shape[1]
    W = make_front(tr)
    sigma = float(np.sqrt(rho * np.asarray(tr, np.float64).var(0).mean()))
    seed_all(seed)
    if DET == 'dart':
        a = CFG['arch']
        net = ScoreNet(D, list(a['hidden']), a['activation'],
                       whitening=W).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=float(TRAIN['lr']),
                               weight_decay=float(TRAIN['weight_decay']))
        cwin = int(SP_CFG['dart_cfar_window'])
    else:
        a = CFG['darts_arch']
        net = _NeighborDenoiser(D, int(a['d_lat']), int(a['K']),
                                list(a['enc_hidden']),
                                list(a['score_hidden']),
                                float(np.sqrt(SP_CFG['darts']['dsm_sigma_rho'])),
                                a['activation'], W).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=float(TRAIN['lr']),
                                weight_decay=float(TRAIN['weight_decay']))
        cwin = int(SP_CFG['darts_cfar_window'] or SP_CFG['k'])
    sched = None
    if TRAIN.get('cosine'):
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=EPOCHS,
            eta_min=float(TRAIN['lr']) * float(TRAIN.get('eta_min_frac', 0.01)))
    clip = float(TRAIN.get('grad_clip', 0.0))
    batch = int(TRAIN['batch_size'])
    X = torch.tensor(np.asarray(tr, np.float32), device=DEVICE)
    N = (torch.tensor(np.asarray(_CACHE['sc']['_tr_nbr'], np.float32),
                      device=DEVICE) if DET == 'darts' else None)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
    P = len(X)
    rundir = os.path.join(CKPT_ROOT, key)
    os.makedirs(rundir, exist_ok=True)

    def score_all(pix, nbr):
        out = []
        with torch.no_grad():
            for i in range(0, len(pix), 4096):
                p = torch.tensor(np.asarray(pix[i:i+4096], np.float32),
                                 device=DEVICE)
                if DET == 'darts':
                    nb = torch.tensor(np.asarray(nbr[i:i+4096], np.float32),
                                      device=DEVICE)
                    out.append(net(p, nb).cpu().numpy())
                else:
                    out.append(net(p).cpu().numpy())
        return np.concatenate(out, 0)

    def evaluate():
        nbr_tr = _CACHE['sc']['_tr_nbr'] if not IS_IID else None
        nbr_te = _CACHE['sc']['_te_nbr'] if not IS_IID else None
        z_tr = score_all(tr, nbr_tr)
        z_te = score_all(planted, nbr_te)
        zb = z_tr.mean(0)
        C = np.cov(z_tr, rowvar=False)
        T = -((z_te - zb) @ s) / np.sqrt(float(s @ C @ s))
        if IS_IID:
            return {'pd': round(float(_pd_at_fa(y, T, 0.1)), 4),
                    'auc': round(float(_auc(y, T)), 4)}, T, None
        Tc = DARTS.local_moment_normalize(
            T, shape, cwin, guard=int(SP_CFG['darts_cfar_guard']),
            cfar_lam=float(SP_CFG['cfar_lam']))
        return {'auc': round(float(auc_safe(y, T)), 4),
                'pd05': round(float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05']), 4),
                'auc_cfar': round(float(auc_safe(y, Tc)), 4),
                'pd05_cfar': round(float(dr_at_fpr(y, Tc,
                                                   fpr_list=(0.05,))['0.05']), 4)
                }, T, Tc

    curve = []
    best = {'_m': -1.0}
    best_state = None
    mkey = 'pd' if IS_IID else 'auc'
    bar = tqdm(range(1, EPOCHS + 1), desc=key, ncols=130, mininterval=5.0,
               file=sys.stdout, ascii=True)
    for ep in bar:
        net.train()
        perm = torch.randperm(P, generator=gen, device=DEVICE)
        for i in range(0, P, batch):
            sel = perm[i:i + batch]
            eps = torch.randn((len(sel), D), generator=gen,
                              device=DEVICE) * sigma
            psi = net(X[sel] + eps, N[sel]) if DET == 'darts' \
                else net(X[sel] + eps)
            loss = ((psi + eps / sigma ** 2) ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
        if sched is not None:
            sched.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            net.eval()
            m, T, Tc = evaluate()
            m['epoch'] = ep
            curve.append(m)
            if m[mkey] > best['_m']:
                best = dict(m); best['_m'] = m[mkey]
                best_state = {k_: v.cpu().clone()
                              for k_, v in net.state_dict().items()}
            torch.save({'w16': {k_: v.half().cpu() for k_, v in
                                net.state_dict().items()}, 'epoch': ep},
                       os.path.join(rundir, f'snap_{ep:06d}.pt'))
            bar.set_postfix_str(f'loss={float(loss.detach()):.3g} '
                                f'{mkey}={m[mkey]:.3f} '
                                f'best={best["_m"]:.3f}@{best["epoch"]}')
    bar.close()
    net.eval()
    m, T, Tc = evaluate()
    save = {'T': T, 'labels': y}
    if Tc is not None:
        save['T_cfar'] = Tc
    np.savez_compressed(os.path.join(rundir, 'scores_final.npz'), **save)
    best.pop('_m', None)
    torch.save({'net_final': {k_: v.cpu() for k_, v in
                              net.state_dict().items()},
                'net_best': best_state, 'best': best, 'config': CFG,
                'rho': rho, 'seed': seed, 'sigma_raw': sigma},
               os.path.join(rundir, 'model.pt'))
    out = {'scenario': SCEN, 'det': DET, 'front': FRONT, 'tag': TAG,
           'n': n, 'rho': rho, 'seed': seed, 'epochs_done': EPOCHS,
           'best': best, 'final': curve[-1], 'curve': curve,
           'config': CFG, 'sec': round(time.time() - t0)}
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    res[key] = out
    json.dump(res, open(OUT_JSON, 'w'), indent=1)
    print(f'[{key}] best={best} final={curve[-1]} ({out["sec"]}s)',
          flush=True)


if __name__ == '__main__':
    rhos = [float(r) for r in CFG['rhos']]
    seeds = [int(s) for s in CFG['seeds']]
    print(f'{SCEN} [{DET}/{FRONT}]: {len(rhos)} rhos x {len(seeds)} seeds, '
          f'{EPOCHS} ep, device={DEVICE}', flush=True)
    for rho in rhos:
        for seed in seeds:
            run_one(rho, seed)
    print('ALL DONE', flush=True)

## MULTI (IID)

In [ ]:
# ================== MULTI (IID) — config & run ==================
CFG = {
  'scenario': 'multi', 'tag': '',
  'det': 'dart',            # dart only (IID has no windows)
  'front': 'std',            # std | zca | robust | none
  'n': 2048,
  'rhos': [0.005, 0.03],          # sequential
  'seeds': [42, 43],         # sequential
  'arch': {'hidden': [128], 'activation': 'relu'},        # DART
  'darts_arch': {'d_lat': 16, 'K': 7, 'enc_hidden': [64, 32],  # DARTS
                 'score_hidden': [128], 'activation': 'relu'},
  'train': {'lr': 5e-4, 'weight_decay': 0.0, 'grad_clip': 1.0,
            'batch_size': 512, 'epochs': 30000,
            'cosine': False, 'eta_min_frac': 0.01},
  'eval_every': 500,
}
import json; json.dump(CFG, open('cfg_multi.json', 'w'), indent=1)
import os; os.environ['CONFIG'] = 'cfg_multi.json'
!python run_play.py

## SINGLE (IID)

In [ ]:
# ================== SINGLE (IID) — config & run ==================
CFG = {
  'scenario': 'single', 'tag': '',
  'det': 'dart',            # dart only (IID has no windows)
  'front': 'std',            # std | zca | robust | none
  'n': 2048,
  'rhos': [0.01, 0.05],          # sequential
  'seeds': [42, 43],         # sequential
  'arch': {'hidden': [128], 'activation': 'relu'},        # DART
  'darts_arch': {'d_lat': 16, 'K': 7, 'enc_hidden': [64, 32],  # DARTS
                 'score_hidden': [128], 'activation': 'relu'},
  'train': {'lr': 5e-4, 'weight_decay': 0.0, 'grad_clip': 1.0,
            'batch_size': 512, 'epochs': 30000,
            'cosine': False, 'eta_min_frac': 0.01},
  'eval_every': 500,
}
import json; json.dump(CFG, open('cfg_single.json', 'w'), indent=1)
import os; os.environ['CONFIG'] = 'cfg_single.json'
!python run_play.py

## PAVIA4 (scene)

In [ ]:
# ================== PAVIA4 (scene) — config & run ==================
CFG = {
  'scenario': 'pavia4', 'tag': '',
  'det': 'dart',            # dart | darts
  'front': 'std',            # std | zca | robust | none
  'target_cls': None,   # None = bitumen sig | 5 = metal sheets
  'theta': 0.15,        # bitumen .15 | metal .075
  'rhos': [0.001, 0.01],          # sequential
  'seeds': [42, 43],         # sequential
  'arch': {'hidden': [128], 'activation': 'relu'},        # DART
  'darts_arch': {'d_lat': 16, 'K': 7, 'enc_hidden': [64, 32],  # DARTS
                 'score_hidden': [128], 'activation': 'relu'},
  'train': {'lr': 5e-4, 'weight_decay': 0.0, 'grad_clip': 1.0,
            'batch_size': 512, 'epochs': 30000,
            'cosine': False, 'eta_min_frac': 0.01},
  'eval_every': 200,
}
import json; json.dump(CFG, open('cfg_pavia4.json', 'w'), indent=1)
import os; os.environ['CONFIG'] = 'cfg_pavia4.json'
!python run_play.py

## SANDIEGO / SD1 (scene)

In [ ]:
# ================== SANDIEGO / SD1 (scene) — config & run ==================
CFG = {
  'scenario': 'sandiego', 'tag': '',
  'det': 'dart',            # dart | darts
  'front': 'std',            # std | zca | robust | none
  'theta': 0.075,
  'rhos': [0.001, 0.01],          # sequential
  'seeds': [42, 43],         # sequential
  'arch': {'hidden': [200], 'activation': 'relu'},        # DART
  'darts_arch': {'d_lat': 16, 'K': 7, 'enc_hidden': [64, 32],  # DARTS
                 'score_hidden': [128], 'activation': 'relu'},
  'train': {'lr': 5e-4, 'weight_decay': 0.0, 'grad_clip': 1.0,
            'batch_size': 512, 'epochs': 30000,
            'cosine': False, 'eta_min_frac': 0.01},
  'eval_every': 200,
}
import json; json.dump(CFG, open('cfg_sandiego.json', 'w'), indent=1)
import os; os.environ['CONFIG'] = 'cfg_sandiego.json'
!python run_play.py

## SANDIEGO2 / SD2 (scene)

In [ ]:
# ================== SANDIEGO2 / SD2 (scene) — config & run ==================
CFG = {
  'scenario': 'sandiego2', 'tag': '',
  'det': 'dart',            # dart | darts
  'front': 'std',            # std | zca | robust | none
  'theta': 0.075,
  'rhos': [0.001, 0.01],          # sequential
  'seeds': [42, 43],         # sequential
  'arch': {'hidden': [200], 'activation': 'relu'},        # DART
  'darts_arch': {'d_lat': 16, 'K': 7, 'enc_hidden': [64, 32],  # DARTS
                 'score_hidden': [128], 'activation': 'relu'},
  'train': {'lr': 5e-4, 'weight_decay': 0.0, 'grad_clip': 1.0,
            'batch_size': 512, 'epochs': 30000,
            'cosine': False, 'eta_min_frac': 0.01},
  'eval_every': 200,
}
import json; json.dump(CFG, open('cfg_sandiego2.json', 'w'), indent=1)
import os; os.environ['CONFIG'] = 'cfg_sandiego2.json'
!python run_play.py

## Summary + archive

In [ ]:
# ---- Summary of everything run so far ----
import json, os
import numpy as np
if os.path.exists('results_play.json'):
    r = json.load(open('results_play.json'))
    for k, v in sorted(r.items()):
        b = {x: y for x, y in v['best'].items() if x != 'epoch'}
        print(f"{k:<55} best={b} @{v['best']['epoch']}  final={v['final']}")
else:
    print('nothing yet')

In [ ]:
# ---- Archive EVERYTHING ----
import shutil, os
!zip -q -r play_results.zip results_play.json ckpt_play cfg_*.json
print(os.path.getsize('play_results.zip')/1e6, 'MB')
from google.colab import files
shutil.copy('play_results.zip', 'play_results_dl.zip')
files.download('play_results_dl.zip')